In [1]:
!python --version

Python 3.10.4


In [15]:
%pip install -q matplotlib nest-asyncio openai pandas python-dotenv safetensors scikit-learn torch tiktoken tqdm 

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [5]:
import sqlite3
conn = sqlite3.connect(f'../train_embeddings.db')

In [29]:
cursor = conn.cursor()
cursor.execute("""
  CREATE TABLE IF NOT EXISTS comments_gpt4_annotated (
    id TEXT PRIMARY KEY,
    comment_text TEXT,
    toxic INTEGER,
    severe_toxic INTEGER,
    obscene INTEGER,
    threat INTEGER,
    insult INTEGER,
    identity_hate INTEGER
);
""")
conn.commit()

In [6]:
cursor = conn.cursor()
cursor.execute("SELECT * FROM comments")
comments = cursor.fetchall()

In [7]:
import openai
import os

# https://wt-dai-eastus2.openai.azure.com/openai/deployments/gpt-4-dev/chat/completions?api-version=2023-03-15-preview
openai.api_key = os.environ["OPENAI_API_KEY"]
# openai.api_key="0ab8f701c7c8405288d206046511fd8c"
# openai.api_type = "azure"
# openai.api_base = "https://wt-dai-eastus2.openai.azure.com/"
# openai.api_version = "2023-03-15-preview"

In [20]:


def annotate_comment(comment, max_attempts=3):
    text = comment[1]
    # make a completion with retry
    attempts = 0
    
    while attempts < max_attempts:
        try:
            response = openai.ChatCompletion.create(
                model="gpt-4",
                messages=[
                    {
                    "role": "system",
                    "content": "You are a content moderator an score user generated comments based on the following criteria:\n\nToxic: very bad, unpleasant, or harmful\nSevere toxic: extremely bad and offensive\nObscene: (of the portrayal or description of sexual matters) offensive or disgusting by accepted standards of morality and decency\nThreat: a statement of an intention to inflict pain, injury, damage, or other hostile action on someone in retribution for something done or not done\nInsult: speak to or treat with disrespect or scornful abuse\nIdentity hate: hatred, hostility, or violence towards members of a race, ethnicity, nation, religion, gender, gender identity, sexual orientation or any other designated sector of society\n\nQ: Hello World\nA: Toxic: 0, Severe Toxic: 0, Obscene: 0, Threat: 0, Insult: 0, Identity Hate: 0\n\nQ: COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK\nA: Toxic: 1, Severe Toxic: 1, Obscene: 1, Threat: 0, Insult: 1, Identity Hate: 0"
                    },
                    {
                    "role": "user",
                    "content": f"Q: {text}\nA:"
                    },
                ],
                temperature=1,
                max_tokens=256,
                top_p=1,
                frequency_penalty=0,
                presence_penalty=0
            )
            return (comment, response)
        except Exception as e:
          print(f"Failed attempt {attempts + 1}: {e}")
          attempts+=1
          

annotate_comment(("id", "text"))

(('id', 'text'),
 <OpenAIObject chat.completion id=chatcmpl-7ouUFQjhHLBfD4GjiphsxID1iHBpf at 0x178a07100> JSON: {
   "id": "chatcmpl-7ouUFQjhHLBfD4GjiphsxID1iHBpf",
   "object": "chat.completion",
   "created": 1692367891,
   "model": "gpt-4-0613",
   "choices": [
     {
       "index": 0,
       "message": {
         "role": "assistant",
         "content": "Please provide the text to moderate."
       },
       "finish_reason": "stop"
     }
   ],
   "usage": {
     "prompt_tokens": 254,
     "completion_tokens": 7,
     "total_tokens": 261
   }
 })

{
  "id": "chatcmpl-7oLVU7MdJabd7fL2KyU4i5klVxRvE",
  "object": "chat.completion",
  "created": 1692233428,
  "model": "gpt-4-0613",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "Toxic: 0, Severe Toxic: 0, Obscene: 0, Threat: 0, Insult: 0, Identity Hate: 0"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 315,
    "completion_tokens": 35,
    "total_tokens": 350
  }
}

In [ ]:
import asyncio
import nest_asyncio
from tqdm import trange

nest_asyncio.apply()
batch_size = 5

for i in trange(0, len(comments), batch_size):
  batch = comments[i:i+batch_size]
  batch_ids = [x[0] for x in batch]
  batch_text = [x[1] for x in batch]

  cursor.execute(
    """
          SELECT id FROM comments_gpt4_annotated WHERE id IN (%s)
    """ % ','.join('?'*len(batch_ids)), batch_ids
  )
  annotated_ids = [x[0] for x in cursor.fetchall()]
  tasks = []
  loop = asyncio.get_event_loop()
  for item in batch:
    id = item[0]
    text = item[1]
    if id not in annotated_ids:
      task = loop.run_in_executor(None, annotate_comment, item)
      tasks.append(task)
      
  for response in await asyncio.gather(*tasks):
    item = response[0]
    completion = response[1]
    try:
      # Update the database
      annotation = completion.choices[0].message.content
      annotation = annotation.split(", ")
      values = [x.split(": ")[1] for x in annotation]

      cursor.execute(
        """
              INSERT OR REPLACE INTO comments_gpt4_annotated (id, comment_text, toxic, severe_toxic, obscene, threat, insult, identity_hate)
              VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (item[0], item[1], values[0], values[1], values[2], values[3], values[4], values[5])
      )
    except:
      # write to file
      with open("failed_comments.txt", "a") as f:
        f.write(id + "\n")

      pass
  conn.commit()

In [10]:
conn.commit()

Reviewing the disagreement in identity hate, there was a discrepancy in 10 rows,
8/10 were incorrectly flagged by the human.
2/10 were not flagged by the ai when they should have been

in a sample of 10 where identiy_hate label was flagged by the robot
10 / 10 were correctly flagged by the robot where the human failed to flag them